In [ ]:
import torch
import numpy as np
def load_results(dirname, dname, gnn, diagram_type, fb_one=False, no_ofst=False,
                 extended_persistence=False, extended_persistence2=False, seed=2):
    fp = f"{dirname}/{dname}/2/8/16_hidden/64_outdim/True_dim1/{diagram_type}_{gnn}_{seed}"
    
    if diagram_type == "forward_backward":
        if fb_one:
            fp += f"_fbone{int(fb_one)}"
        if extended_persistence:
            fp += f"_extpers{int(extended_persistence)}"
        if extended_persistence2:
            fp += f"_extpers2{int(extended_persistence2)}"
        if no_ofst:
            fp += f"_noofst{int(no_ofst)}"

    # print(f"Loading results from: {fp}.results")
    results = torch.load(f"{fp}.results")
    return results


def make_table(dirname, gnn, mode,seeds):
    """
    mode = "max"  → return only max
    mode = "es_max"  → return only val_best
    """
    datasets = ["NCI109", "PROTEINS", "IMDB-BINARY","NCI1","ZINC", "ogbg-molhiv"]
    methods = [
        ("standard", False, False, False, False),
        ("rephine", False, False, False, False),
        ("forward_only", False, False, False, False),  # forward only
        ("backward_only", False, False, False, False),  # backward only
        # ("forward_backward", False, False, True, False),  # extended persistence
        # ("forward_backward", False, False, False, True),  # extended persistence 2
        # ("forward_backward", False, True, True, False),  # extended persistence + no_ofst
        # ("forward_backward", False, True, False, True),  # extended persistence 2 + no_ofst     
        ("forward_backward", True, True, False, False),   # fb_one + no_ofst
        ("forward_backward", True, False, False, False),  # fb_one only
        ("forward_backward", False, False, False, False),  # plain forward_backward
    ]
    # seeds = [1, 2, 3, 4, 5]
    
    table = []
    for dname in datasets:
        row = []
        for diagram_type, fb_one, no_ofst, extended_persistence, extended_persistence2 in methods:
            accs = []
            for seed in seeds:
                try:
                    res = load_results(dirname, dname, gnn, diagram_type,
                                       fb_one=fb_one, no_ofst=no_ofst, 
                                       extended_persistence=extended_persistence,
                                       extended_persistence2=extended_persistence2,
                                       seed=seed)
                    test_acc = res['test_accuracies']
                    val_best = res['val_accuracies'].argmax()
                    # print(f"{dname} | {diagram_type} | seed {seed} | test_acc: {test_acc} | val_best: {val_best}")

                    if mode == "max":
                        accs.append(test_acc.max().item())
                    elif mode == "es_max":
                        accs.append(test_acc[val_best].item())

                except FileNotFoundError:
                    continue

            if len(accs) > 0:
                row.append((np.mean(accs), np.std(accs)))
            else:
                row.append(("N/A", "N/A"))

        table.append((dname, row))
    return table

    # table = []
    # for dname in datasets:
    #     row = []
    #     for diagram_type, fb_one, no_ofst in methods:
    #         try:
    #             res = load_results(dirname, dname, gnn, diagram_type, fb_one, no_ofst)
    #             test_acc = res['test_accuracies']
    #             max_acc = test_acc.max().item()
    #             val_best_test_acc = test_acc[res['val_accuracies'].argmax()].item()
          
    #             if mode == "max":
    #                 row.append(max_acc)
    #             elif mode == "es_max":
    #                 row.append(val_best_test_acc)
    #         except FileNotFoundError:
    #             row.append("N/A")
    #     table.append((dname, row))
    # return table


def print_table(table, gnn, mode):
    # First 4 individual columns, then "Ours" (max of last 3)
    headers = ["standard", "rephine", "forward_only", "backward_only", "Ours"]

    print(f"\n=== Results for {gnn.upper()} ({mode}) ===")
    print(f"{'Dataset':<14} " + " | ".join(f"{h:<20}" for h in headers))
    print("-" * 110)

    for dname, row in table:

        # ----- Columns 0–3: print mean ± std directly -----
        vals = []
        for (mean, std) in row[:4]:
            if isinstance(mean, float):
                vals.append(f"{mean:.4f} ± {std:.4f}")
            else:
                vals.append("N/A")

        # ----- Column 4: best among last 3 variants (indices 4,5,6) -----
        our_variants = [(m, s) for (m, s) in row[4:] if isinstance(m, float)]

        if our_variants:
            best_mean, best_std = max(our_variants, key=lambda x: x[0])
            vals.append(f"{best_mean:.4f} ± {best_std:.4f}")
        else:
            vals.append("N/A")

        # ----- Print final row -----
        print(f"{dname:<14} " + " | ".join(f"{v:<20}" for v in vals))


# def print_table(table, gnn, mode="max"):
#     headers = ["standard", "rephine", "Ours"]
#     print(f"\n=== Results for {gnn.upper()} ({mode}) ===")
#     print(f"{'Dataset':<12} " + " | ".join(f"{h:<14}" for h in headers))
#     print("-" * 60)
#     for dname, row in table:
#         vals = []
#         # Baselines
#         for entry in row[:2]:
#             if isinstance(entry, (int, float)):
#                 vals.append(f"{entry:.4f}")
#             else:
#                 vals.append(str(entry))
#         our_variants = [x for x in row[2:] if isinstance(x, (int, float))]
#         if our_variants:
#             vals.append(f"{max(our_variants):.4f}")
#         else:
#             vals.append("N/A")
#         print(f"{dname:<12} " + " | ".join(f"{v:<14}" for v in vals))



In [76]:
dirname = "results/main"
gnn = "gcn"
mode = "es_max"
print_table(make_table(dirname, gnn, mode,seeds=[1,2,3]), gnn, mode)


=== Results for GCN (es_max) ===
Dataset        standard             | rephine              | forward_only         | backward_only        | Ours                
--------------------------------------------------------------------------------------------------------------
NCI109         0.7659 ± 0.0132      | 0.7950 ± 0.0011      | 0.7191 ± 0.0052      | 0.7458 ± 0.0071      | 0.7587 ± 0.0089     
PROTEINS       0.7054 ± 0.0073      | 0.6875 ± 0.0253      | 0.6935 ± 0.0183      | 0.7054 ± 0.0146      | 0.7232 ± 0.0146     
IMDB-BINARY    0.6500 ± 0.0163      | 0.7000 ± 0.0082      | 0.6267 ± 0.0330      | 0.6433 ± 0.0330      | 0.6800 ± 0.0216     
NCI1           0.7843 ± 0.0098      | 0.7891 ± 0.0080      | 0.7559 ± 0.0100      | 0.7624 ± 0.0174      | 0.7867 ± 0.0169     
ZINC           -0.4880 ± 0.0207     | -0.4588 ± 0.0093     | -0.8599 ± 0.0129     | -0.8671 ± 0.0086     | -0.4389 ± 0.0055    
ogbg-molhiv    0.7512 ± 0.0068      | 0.7540 ± 0.0053      | 0.7102 ± 0.0218      | 0.7

In [77]:
dirname = "results/main"
gnn = "gin"
mode = "es_max"
print_table(make_table(dirname, gnn, mode,seeds=[1,2,3]), gnn, mode)


=== Results for GIN (es_max) ===
Dataset        standard             | rephine              | forward_only         | backward_only        | Ours                
--------------------------------------------------------------------------------------------------------------
NCI109         0.7676 ± 0.0040      | 0.7789 ± 0.0119      | 0.7700 ± 0.0103      | 0.7635 ± 0.0050      | 0.7789 ± 0.0187     
PROTEINS       0.6935 ± 0.0183      | 0.6994 ± 0.0276      | 0.7024 ± 0.0295      | 0.7054 ± 0.0219      | 0.7351 ± 0.0111     
IMDB-BINARY    0.6867 ± 0.0125      | 0.7067 ± 0.0094      | 0.7467 ± 0.0047      | 0.7433 ± 0.0094      | 0.7200 ± 0.0216     
NCI1           0.7924 ± 0.0174      | 0.7875 ± 0.0255      | 0.7672 ± 0.0113      | 0.7575 ± 0.0094      | 0.8127 ± 0.0000     
ZINC           -0.4324 ± 0.0098     | -0.4144 ± 0.0108     | -0.6173 ± 0.0138     | -0.6088 ± 0.0044     | -0.4011 ± 0.0075    
ogbg-molhiv    0.7434 ± 0.0457      | 0.7288 ± 0.0215      | 0.7000 ± 0.0311      | 0.7